[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/45_full_grpo_loss.ipynb)

# 🔴 Hard: Full GRPO Loss

Implement a more complete **GRPO objective** with three pieces:

1. group-wise normalized advantages from scalar rewards
2. a PPO-style clipped importance ratio against `old_logps`
3. a KL penalty to a fixed reference policy `ref_logps`

A common token-level form is:

$$
L = -
rac{1}{B}\sum_i 
rac{1}{|y_i|}\sum_t m_{i,t}\Big(\min(r_{i,t} A_i, 	ext{clip}(r_{i,t}, 1-\epsilon, 1+\epsilon) A_i) - eta\,\mathrm{KL}_{i,t}\Big)
$$

where `m_{i,t}` is a completion mask and
$$
r_{i,t} = \exp(\log \pi_	heta - \log \pi_{old})
$$

### Signature
```python
from torch import Tensor

def full_grpo_loss(logps: Tensor, old_logps: Tensor, ref_logps: Tensor,
                   rewards: Tensor, group_ids: Tensor, completion_mask: Tensor,
                   clip_ratio: float = 0.2, beta: float = 0.1, eps: float = 1e-5) -> Tensor:
    # logps / old_logps / ref_logps: (B, T) token log-probs
    # rewards: (B,) scalar reward per sampled completion
    # group_ids: (B,) same id means same prompt / group
    # completion_mask: (B, T) with 1 for valid completion tokens
    # returns: scalar loss
```

### Rules
- Normalize rewards **within each group**
- Detach old policy, reference policy, and advantages
- Ignore padded tokens using `completion_mask`
- Average over valid tokens per sequence before averaging over the batch


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
from torch import Tensor


In [30]:
# ✏️ YOUR IMPLEMENTATION HERE

def full_grpo_loss(logps: Tensor, old_logps: Tensor, ref_logps: Tensor,
                   rewards: Tensor, group_ids: Tensor, completion_mask: Tensor,
                   clip_ratio: float = 0.2, beta: float = 0.1, eps: float = 1e-5) -> Tensor:
    # pass
    old_logps = old_logps.detach()
    ref_logps = ref_logps.detach()
    rewards = rewards.detach()

    uni_group_ids = torch.unique(group_ids)
    advantages = torch.empty_like(rewards) # b 
    for gid in uni_group_ids:
        mask = group_ids == gid
        r_g = rewards[mask]
        mean_g = r_g.mean()
        std_g = r_g.std(unbiased=False)
        advantages[mask] = (r_g - mean_g) / (std_g + eps)
    advantages = advantages.unsqueeze(1)
    
    new_old_ratio = (logps - old_logps).exp()
    new_old_ratio_clipped = torch.clip(new_old_ratio, 1-clip_ratio, 1+clip_ratio)
    min_r_adv = torch.min(new_old_ratio*advantages, new_old_ratio_clipped*advantages)

    kl = torch.zeros_like(logps)
    delta = ref_logps - logps
    kl = torch.exp(delta) - delta - 1.0

    obj = min_r_adv - beta * kl
    
    # token_counts = completion_mask.sum(dim=-1).clamp_min(1.0) # 每个样本的有效token (b)
    # obj = (obj * completion_mask).sum(dim=-1)  / token_counts

    return -obj.mean()



    # new_ref_ratio = 
    
    
    
    
    


In [31]:
# 🧪 Debug
logps = torch.tensor([[-0.2, -0.1, -0.3], [-0.6, -0.5, -0.4], [-0.3, -0.8, -1.0], [-0.9, -1.1, -1.2]])
old_logps = torch.tensor([[-0.3, -0.2, -0.4], [-0.4, -0.4, -0.4], [-0.5, -0.6, -0.9], [-0.7, -1.0, -1.3]])
ref_logps = torch.tensor([[-0.25, -0.15, -0.35], [-0.55, -0.45, -0.45], [-0.35, -0.7, -0.95], [-0.8, -1.0, -1.1]])
rewards = torch.tensor([1.0, 0.8, 0.3, 0.1])
group_ids = torch.tensor([0, 0, 1, 1])
completion_mask = torch.tensor([[1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1]], dtype=torch.float32)
print('Loss:', full_grpo_loss(logps, old_logps, ref_logps, rewards, group_ids, completion_mask))


Loss: tensor(-0.0570)


In [27]:
# ✅ SUBMIT
from torch_judge import check
check('full_grpo_loss')



🧪 Testing: Full GRPO (Clipped + KL) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Basic shape & type (9.2ms)
  ✅ [2/5] Numeric check vs reference (1.2ms)
  ✅ [3/5] Zero group advantages leave only KL term (0.5ms)
  ❌ [4/5] Padding mask excludes padded tokens
     Masked tokens should not affect the loss
  ✅ [5/5] Gradient flows to current logps only (1.2ms)
──────────────────────────────────────────────────
  📊 4/5 tests passed.
  Keep going! Use hint("full_grpo_loss") if you're stuck.

